# Notebook 01: For Loops, Iteration Protocols, and Numerical Methods

This notebook explores Python iteration mechanics, loop control flow, and their application to fundamental numerical approximation algorithms:
1. **Iteration Protocols & Control Flow:** `range()`, `break`, `continue`, `enumerate()`, and `zip()`.
2. **Gregory-Leibniz Series for $\pi$:** $O(1/n)$ slow alternating series.
3. **Nilakantha Series for $\pi$:** $O(1/n^3)$ accelerated cubic polynomial series.
4. **Euler's Number ($e$):** Compound limit definition and Taylor series expansion.
5. **Newton's Method (Babylonian Method):** Quadratic convergence square root computation.


In [ ]:
import json
import math
from pathlib import Path

# Load reference numerical benchmarks
DATA_PATH = Path("../data/numerical_series_benchmarks.json")
with DATA_PATH.open("r", encoding="utf-8") as f:
    benchmarks = json.load(f)

print("Loaded benchmarks successfully. Available sections:", list(benchmarks.keys()))

## 1. Loop Control Flow: `range`, `break`, and `continue`

The `for` loop in Python iterates over any iterable sequence. `range(start, stop, step)` generates integers on-demand with $O(1)$ memory.


In [ ]:
# Basic range iteration
evens = []
for i in range(0, 10, 2):
    evens.append(i)
assert evens == [0, 2, 4, 6, 8]

# Early exit with break
found = None
for val in range(1, 100):
    if val**2 > 50:
        found = val
        break
assert found == 8

# Skipping items with continue
odds = []
for i in range(10):
    if i % 2 == 0:
        continue
    odds.append(i)
assert odds == [1, 3, 5, 7, 9]

print("Loop control assertions passed!")

## 2. Iteration Utilities: `enumerate` and `zip`

`enumerate()` yields `(index, item)` pairs without manual counter indexing. `zip()` aggregates elements from multiple iterables element-wise.


In [ ]:
fruits = ["apple", "banana", "cherry"]
indexed = []
for idx, item in enumerate(fruits):
    indexed.append((idx, item))
assert indexed == [(0, "apple"), (1, "banana"), (2, "cherry")]

names = ["alpha", "beta", "gamma"]
values = [0.1, 0.01, 0.001]
paired = []
for name, val in zip(names, values, strict=True):
    paired.append(f"{name}={val}")
assert paired == ["alpha=0.1", "beta=0.01", "gamma=0.001"]

print("enumerate and zip assertions passed!")

## 3. Estimating $\pi$: Gregory-Leibniz vs. Nilakantha Series

### Gregory-Leibniz Formula:
$$\pi pprox 4 \sum_{i=1}^n rac{(-1)^{i+1}}{2i - 1}$$

### Nilakantha Series:
$$\pi pprox 3 + 4 \sum_{i=0}^{n-1} rac{(-1)^i}{(2i+2)(2i+3)(2i+4)}$$


In [ ]:
def estimate_pi_gregory_leibniz(n: int) -> float:
    """Estimate Pi using the Gregory-Leibniz alternating series."""
    pi_estimate = 0.0
    for i in range(1, n + 1):
        pi_estimate += ((-1) ** (i + 1)) / (2 * i - 1)
    return 4.0 * pi_estimate


def estimate_pi_nilakantha(n: int) -> float:
    """Estimate Pi using the Nilakantha accelerated series."""
    pi_estimate = 0.0
    for i in range(n):
        denom = (2 * i + 2) * (2 * i + 3) * (2 * i + 4)
        pi_estimate += ((-1) ** i) / denom
    return 3.0 + 4.0 * pi_estimate


# Benchmark comparison at n = 1000
pi_gl = estimate_pi_gregory_leibniz(1000)
pi_nil = estimate_pi_nilakantha(1000)
true_pi = math.pi

err_gl = abs(pi_gl - true_pi)
err_nil = abs(pi_nil - true_pi)

print(f"Gregory-Leibniz (n=1000): {pi_gl:.8f} (Error: {err_gl:.2e})")
print(f"Nilakantha (n=1000):      {pi_nil:.12f} (Error: {err_nil:.2e})")

assert err_nil < err_gl, "Nilakantha must converge significantly faster than Gregory-Leibniz"
assert err_nil < 1e-9, "Nilakantha error must be below 1e-9 at n=1000"

## 4. Euler's Number ($e$): Limit vs. Taylor Series Expansion

### Limit Formulation:
$$e = \lim_{n 	o \infty} \left(1 + rac{1}{n}ight)^n$$

### Taylor Series Expansion:
$$e = \sum_{k=0}^\infty rac{1}{k!} = 1 + rac{1}{1!} + rac{1}{2!} + \dots + rac{1}{n!}$$


In [ ]:
def estimate_e_limit(n: int) -> float:
    """Estimate Euler's number via the compound interest limit."""
    return (1.0 + 1.0 / n) ** n


def compute_factorial(n: int) -> int:
    """Iteratively compute factorial of non-negative integer n."""
    fact = 1
    for i in range(2, n + 1):
        fact *= i
    return fact


def estimate_e_taylor(n_terms: int) -> float:
    """Estimate Euler's number via Maclaurin/Taylor series."""
    e_val = 1.0
    for k in range(1, n_terms + 1):
        e_val += 1.0 / compute_factorial(k)
    return e_val


# Compare limit vs Taylor series
e_lim_100k = estimate_e_limit(100_000)
e_taylor_10 = estimate_e_taylor(10)
true_e = math.e

print(f"Limit (n=100,000): {e_lim_100k:.7f} (Error: {abs(e_lim_100k - true_e):.2e})")
print(f"Taylor (n=10):     {e_taylor_10:.10f} (Error: {abs(e_taylor_10 - true_e):.2e})")

assert abs(e_taylor_10 - true_e) < 1e-7, "10 Taylor terms must achieve <1e-7 error"

## 5. Square Root Approximation via Newton's Method

Newton-Raphson method applied to $f(x) = x^2 - N = 0$:
$$x_{n+1} = rac{1}{2} \left(x_n + rac{N}{x_n}ight)$$


In [ ]:
def compute_square_root(
    n_val: float, max_loops: int = 10, tolerance: float = 1e-12
) -> tuple[float, list[float]]:
    """Compute square root using the Babylonian / Newton-Raphson method."""
    if n_val < 0:
        raise ValueError("Cannot compute real square root of negative number.")
    if n_val == 0:
        return 0.0, [0.0]

    x_curr = n_val / 2.0
    trace = [x_curr]
    for _ in range(max_loops):
        x_next = 0.5 * (x_curr + n_val / x_curr)
        trace.append(x_next)
        if abs(x_next - x_curr) < tolerance:
            break
        x_curr = x_next
    return x_curr, trace


# Validate for N = 9, 2, 144
for n_test, expected in [(9.0, 3.0), (2.0, math.sqrt(2)), (144.0, 12.0)]:
    root, trace = compute_square_root(n_test)
    print(f"sqrt({n_test}) = {root:.10f} (Converged in {len(trace) - 1} iterations)")
    assert math.isclose(root, expected, rel_tol=1e-10)

print("All numerical methods tests passed successfully!")